In [1]:
import pandas as pd
import numpy as np
import glob
from scipy.interpolate import griddata
from datetime import datetime
import math
import pandas as pd
import numpy as np
from pathlib import Path
import glob
from scipy.interpolate import griddata
import os
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from scipy.interpolate import griddata
from PIL import Image


CONFIG


In [2]:

# CONFIG
ERA5_PATH = Path("/home/sangonvi/Cefet/repositories/atmoseer/data/reanalysis/cds/era5/pressure")
RADAR_PATH = Path("/home/sangonvi/Cefet/repositories/atmoseer/data/radar_sumare")
RADAR_CACHE = Path("radar_cache")

RADAR_RES_KM = 2
ERA5_RES_DEG = 0.25
ONE_DEG_LATLON_IN_KM = 111.32

OUTPUT_DIR = Path("dataset")
PATCH_SIZE = 32
STRIDE = 16

TIME_START = "2024-01-22"
TIME_END   = "2024-01-23"

ERA5_VARS = ["v", "u"]  # ajuste

N_THREADS = 12      # I/O
N_PROCESSES = 6     # CPU (interpolação + patches)

OUTPUT_DIR.mkdir(exist_ok=True)

In [3]:
print(str(RADAR_PATH))
print('/home/sangonvi/Cefet/repositories/atmoseer/data/radar_sumare/2024/01/22/2024_01_22_00_00.png')

/home/sangonvi/Cefet/repositories/atmoseer/data/radar_sumare
/home/sangonvi/Cefet/repositories/atmoseer/data/radar_sumare/2024/01/22/2024_01_22_00_00.png


1. LOAD ERA5 (MULTI-MONTH)


In [4]:
def load_era5_file(f):
    df = pd.read_parquet(f)
    if "time" not in df.columns:
        df = df.reset_index()
    
    df["time"] = pd.to_datetime(df["valid_time"])
    return df

era5_files = list(ERA5_PATH.rglob("*.parquet"))
with ThreadPoolExecutor(max_workers=N_THREADS) as ex:
    dfs = list(ex.map(load_era5_file, era5_files))

era5 = pd.concat(dfs, ignore_index=True)
era5 = era5.sort_values("time")

In [5]:
era5["time"] = pd.to_datetime(era5["valid_time"])

era5 = era5[
    (era5["time"] >= TIME_START) &
    (era5["time"] <= TIME_END)
]

era5 = era5.sort_values("time")

print("ERA5 loaded:", era5.shape)

ERA5 loaded: (1050, 16)


In [6]:
era5.columns

Index(['index', 'valid_time', 'pressure_level', 'latitude', 'longitude', 'u',
       'v', 'r', 'q', 't', 'w', 'crwc', 'number', 'expver', 'day', 'time'],
      dtype='object')

In [7]:
def rgb_distance(c1, c2):
    return math.sqrt((c1[0] - c2[0]) ** 2 + (c1[1] - c2[1]) ** 2 + (c1[2] - c2[2]) ** 2)


def interpolate_value(rgb, legend_colors, legend_values):
    if rgb == (0, 0, 0):
        return 0

    distances = [rgb_distance(rgb, lc) for lc in legend_colors]

    min_idx1 = distances.index(min(distances)) 
    distances[min_idx1] = float(
        "inf"
    )  
    min_idx2 = distances.index(min(distances)) 

    # Obter as cores e valores correspondentes
    c1, c2 = legend_colors[min_idx1], legend_colors[min_idx2]
    v1, v2 = legend_values[min_idx1], legend_values[min_idx2]

    # Calcular o peso de interpolação (t)
    dist_c1_c2 = rgb_distance(c1, c2)
    dist_c1_rgb = rgb_distance(c1, rgb)
    t = dist_c1_rgb / dist_c1_c2 if dist_c1_c2 != 0 else 0

    # Interpolar o valor
    interpolated_value = v1 + t * (v2 - v1)
    return interpolated_value


def get_radar_data(inicio, fim, frequencia, latitude, longitude):
    latitude = float(latitude)
    longitude = float(longitude)

    pos_sumare = (-22.955139, -43.248278)
    legend_values = [50, 45, 40, 35, 30, 25, 20, 0]
    legend_colors = [
        (197, 0, 197),  # Magenta
        (227, 6, 5),  # Vermelho
        (255, 112, 0),  # Laranja
        (195, 230, 0),  # Amarelo
        (4, 85, 4),  # Amarelo
        (19, 122, 19),  # Verde escuro
        (0, 167, 12),  # Verde claro
        (0, 0, 0),  # Vazio
    ]

    # Definir datas inicial e final
    data_inicial = inicio + " 00:00:00"
    data_final = fim + " 00:00:00"

    datas = pd.date_range(start=data_inicial, end=data_final, freq=frequencia)

    # Criar DataFrame com coluna "datahora"
    radar_data = pd.DataFrame({"time": datas})
    radar_data["reflect"] = np.nan

    count = 0
    while count < len(radar_data):
        datetime_str = radar_data.iloc[count]["time"].strftime("%Y-%m-%d %H:%M:%S")

        year = datetime_str[0:4]
        month = datetime_str[5:7]
        day = datetime_str[8:10]
        hour = datetime_str[11:13]
        minute = datetime_str[14:16]
        file = year + "_" + month + "_" + day + "_" + hour + "_" + minute + ".png"
        file_path = str(RADAR_PATH) + "/" + year + "/" + month + "/" + day + "/" + file

        if os.path.exists(file_path):
            try:
                img = Image.open(file_path)
                pos_sumare_img = (img.height / 2, img.width / 2)

                dify = pos_sumare_img[0] / pos_sumare[0]
                difx = pos_sumare_img[1] / pos_sumare[1]

                posx = pos_sumare[1] - ((longitude - pos_sumare[1]) * 32.5)
                valorx = posx * difx
                posy = pos_sumare[0] + ((latitude - pos_sumare[0]) * 19.5)
                valory = posy * dify

                rgb_im = img.convert("RGB")
                r, g, b = rgb_im.getpixel((valorx, valory))
                radar_data.at[count, "reflect"] = interpolate_value((r, g, b), legend_colors, legend_values)
            except Exception as e:
                print("Mensagem de erro:", str(e))  # como string

        count += 1

    return radar_data

In [8]:
lon = np.sort(np.array(era5["longitude"].unique()))
lat = np.sort(np.array(era5["latitude"].unique()))
print("Longitude points:", lon)
print("Latitude points:", lat)

Lon, Lat = np.meshgrid(lon, lat)

radar_lat_lon_interval = RADAR_RES_KM / ONE_DEG_LATLON_IN_KM
print("Radar lat/lon interval (degrees):", radar_lat_lon_interval)
number_lat_points = int((era5["latitude"].max() - era5["latitude"].min()) / radar_lat_lon_interval) + 1
number_lon_points = int((era5["longitude"].max() - era5["longitude"].min()) / radar_lat_lon_interval) + 1
print("Number of radar points - Latitude:", number_lat_points, "Longitude:", number_lon_points)

radar_lat = np.linspace(-23.5, -22.25, number_lat_points)
radar_lon = np.linspace(-44.0, -42.5, number_lon_points)

RadarLon, RadarLat = np.meshgrid(radar_lon, radar_lat)

Longitude points: [-44.   -43.75 -43.5  -43.25 -43.   -42.75 -42.5 ]
Latitude points: [-23.5  -23.25 -23.   -22.75 -22.5  -22.25]
Radar lat/lon interval (degrees): 0.017966223499820338
Number of radar points - Latitude: 70 Longitude: 84


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# =============================
# CONFIG
# =============================
CACHE_DIR = Path("radar_cache")
CACHE_DIR.mkdir(exist_ok=True)

MAX_WORKERS = 10  # ajuste (8–20 ideal)

lock = threading.Lock()
counter = {"done": 0}

# =============================
# CACHE PATH
# =============================
def get_cache_path(lat, lon):
    return CACHE_DIR / f"radar_lat_{lat:.4f}_lon_{lon:.4f}.parquet"

# =============================
# DOWNLOAD + CACHE
# =============================
def process_point(lat, lon):
    path = get_cache_path(lat, lon)

    # -------------------------
    # já existe → skip
    # -------------------------
    if path.exists():
        with lock:
            counter["done"] += 1
            print(f"[{counter['done']}] CACHE OK ({lat:.3f}, {lon:.3f})")
        return (lat, lon, "cached")

    try:
        # -------------------------
        # download
        # -------------------------
        df = get_radar_data(inicio=TIME_START, fim=TIME_END, frequencia='2T',latitude=lat, longitude=lon)

        df["time"] = pd.to_datetime(df["time"])
        df = df.set_index("time")

        # agregação 2min → 1h
        df = df.resample("1H").max()

        # -------------------------
        # salvar
        # -------------------------
        df.reset_index().to_parquet(path, compression="snappy")

        with lock:
            counter["done"] += 1
            print(f"[{counter['done']}] DOWNLOADED ({lat:.3f}, {lon:.3f})")

        return (lat, lon, "downloaded")

    except Exception as e:
        with lock:
            counter["done"] += 1
            print(f"[{counter['done']}] ERROR ({lat:.3f}, {lon:.3f}) -> {e}")
        return (lat, lon, "error")

# =============================
# PARALLEL EXECUTION
# =============================
print("🚀 Starting parallel radar download...")

tasks = [(lat, lon) for lat in radar_lat for lon in radar_lon]
total = len(tasks)

print(f"Total points: {total}")

results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(process_point, lat, lon) for lat, lon in tasks]

    for future in as_completed(futures):
        results.append(future.result())

print("✅ Download finished")

In [10]:
def load_radar_cache_file(f):
    df = pd.read_parquet(f)
    df["time"] = pd.to_datetime(df["time"])
    df = df.set_index("time")
    
    # extrair lat/lon do nome
    name = f.stem
    parts = name.split("_")
    lat = float(parts[2])
    lon = float(parts[4])
    
    return (lat, lon, df)

files = list(RADAR_CACHE.glob("*.parquet"))

radar_data = {}

with ThreadPoolExecutor(max_workers=N_THREADS) as ex:
    for lat, lon, df in ex.map(load_radar_cache_file, files):
        radar_data[(lat, lon)] = df

In [11]:
print("Total de pontos no radar_data:", len(radar_data))
valid = sum(1 for v in radar_data.values() if v is not None)
invalid = sum(1 for v in radar_data.values() if v is None)

print("Pontos válidos:", valid)
print("Pontos inválidos:", invalid)

for (lat, lon), df in radar_data.items():
    print("Lat:", lat, "Lon:", lon)
    print("Tipo:", type(df))
    
    if df is not None:
        print("Shape:", df.shape)
        print("Colunas:", df.columns)
        print(df.head())
    else:
        print("Sem dados")
    
    break  # só um exemplo

values = []


Total de pontos no radar_data: 5880
Pontos válidos: 5880
Pontos inválidos: 0
Lat: -23.2464 Lon: -42.6627
Tipo: <class 'pandas.core.frame.DataFrame'>
Shape: (25, 1)
Colunas: Index(['reflect'], dtype='object')
                     reflect
time                        
2024-01-22 00:00:00      0.0
2024-01-22 01:00:00      0.0
2024-01-22 02:00:00      0.0
2024-01-22 03:00:00      0.0
2024-01-22 04:00:00      0.0


In [12]:
def process_time_step(t):
    era5_t = era5[era5["time"] == t]
    
    if len(era5_t) == 0:
        return None
    
    try:
        points = era5_t[["latitude", "longitude"]].values
        
        channels = []
        for var in ERA5_VARS:
            values = era5_t[var].values
            
            interp = griddata(
                points,
                values,
                (RadarLat, RadarLon),
                method="linear"
            )
            channels.append(interp)
        
        X_t = np.stack(channels, axis=0)
        
        # radar grid
        Y_t = np.full(RadarLat.shape, np.nan)
        
        for i, lat in enumerate(radar_lat):
            for j, lon in enumerate(radar_lon):
                df = radar_data.get((lat, lon))
                
                if df is not None and t in df.index:
                    Y_t[i, j] = df.loc[t].values[0]
        
        if np.isnan(Y_t).all():
            return None
        
        return (t, X_t, Y_t)
    
    except:
        return None

In [13]:
times = sorted(era5["time"].unique())
results = []

with ProcessPoolExecutor(max_workers=N_PROCESSES) as ex:
    futures = [ex.submit(process_time_step, t) for t in times]
    
    for f in as_completed(futures):
        r = f.result()
        if r is not None:
            results.append(r)

In [ ]:
def create_patches(sample):
    t, X_t, Y_t = sample
    
    patches = []
    
    H, W = Y_t.shape
    
    for i in range(0, H - PATCH_SIZE + 1, STRIDE):
        for j in range(0, W - PATCH_SIZE + 1, STRIDE):
            
            xp = X_t[:, i:i+PATCH_SIZE, j:j+PATCH_SIZE]
            yp = Y_t[i:i+PATCH_SIZE, j:j+PATCH_SIZE]
            
            valid_ratio = np.mean(~np.isnan(yp))

            if valid_ratio < 0.3:
                continue

            mask = ~np.isnan(yp)
            yp = np.nan_to_num(yp, nan=0.0)
            
    
            patches.append((xp, yp))
    
    return patches

In [ ]:
print("Building dataset...")

times = sorted(era5["time"].unique())

sample_id = 0

for t in times:
    print("Time:", t)

    era5_t = era5[era5["time"] == t]

    if len(era5_t) == 0:
        continue

    X_t = interpolate_fast(era5_t)
    Y_t = build_radar_grid(t)

    if np.isnan(Y_t).all():
        continue

    H, W = Y_t.shape

    for i in range(0, H - PATCH_SIZE + 1, STRIDE):
        for j in range(0, W - PATCH_SIZE + 1, STRIDE):

            xp = X_t[:, i:i+PATCH_SIZE, j:j+PATCH_SIZE]
            yp = Y_t[i:i+PATCH_SIZE, j:j+PATCH_SIZE]

            valid_ratio = np.mean(~np.isnan(yp))

            if valid_ratio < 0.3:
                continue

            mask = ~np.isnan(yp)
            yp = np.nan_to_num(yp, nan=0.0)

            np.savez_compressed(
                OUTPUT_DIR / "train" / f"sample_{sample_id:06d}.npz",
                input=xp.astype(np.float32),
                target=yp[None, ...].astype(np.float32),
                mask=mask[None, ...].astype(np.float32)
            )

            sample_id += 1

print("Dataset ready:", sample_id)